In [7]:
import os, glob
import numpy as np, pandas as pd

os.makedirs("../clean", exist_ok=True)
cleaning_log = []
def log_step(op, src, aff, bef, aft, rsn):
    cleaning_log.append({"step": len(cleaning_log)+1, "source": src, "operation": op,
                         "rows_affected": int(aff), "before_count": bef, "after_count": aft, "reason": rsn})

# 1. Load the 3 cleaned datasets from interim/
usgs_noaa = pd.read_parquet("../interim/usgs_noaa_cleaned_2026-10-21.parquet")
worldbank = pd.read_parquet("../interim/worldbank_cleaned_2026-10-28.parquet")
wikipedia = pd.read_parquet("../interim/wikipedia_cleaned_2026-10-28.parquet")

# 2. Match Wikipedia to USGS-NOAA Mainshocks
usgs_noaa["date_dt"] = pd.to_datetime(usgs_noaa["date"])
wikipedia["date_dt"] = pd.to_datetime(wikipedia["date"])
used_wiki, wiki_deaths = set(), []

for idx, r in usgs_noaa.sort_values("mag", ascending=False).iterrows():
    cands = wikipedia[((wikipedia["date_dt"] - r["date_dt"]).abs().dt.days <= 1) & 
                      ((wikipedia["mag"] - r["mag"]).abs() <= 0.4) & 
                      (~wikipedia.index.isin(used_wiki))].copy()
    if not cands.empty:
        cands["mag_diff"] = (cands["mag"] - r["mag"]).abs()
        best = cands.sort_values("mag_diff").iloc[0]
        used_wiki.add(best.name)
        wiki_deaths.append((idx, best["wiki_deaths"]))
    else:
        wiki_deaths.append((idx, np.nan))

usgs_noaa["wiki_deaths"] = pd.Series(dict(wiki_deaths)).reindex(usgs_noaa.index).fillna(0.0)
usgs_noaa.rename(columns={"deaths_clean": "noaa_deaths"}, inplace=True)

# 3. Resolve NOAA vs. Wikipedia death disagreements
both_pos = (usgs_noaa["noaa_deaths"] > 0) & (usgs_noaa["wiki_deaths"] > 0)
disagree_mask = both_pos & (usgs_noaa["noaa_deaths"] != usgs_noaa["wiki_deaths"])
one_src = ((usgs_noaa["noaa_deaths"] > 0) ^ (usgs_noaa["wiki_deaths"] > 0))

usgs_noaa["deaths_disagree"] = disagree_mask
usgs_noaa["deaths_final"] = np.where(both_pos, np.round((usgs_noaa["noaa_deaths"] + usgs_noaa["wiki_deaths"]) / 2.0, 1),
                                     np.maximum(usgs_noaa["noaa_deaths"], usgs_noaa["wiki_deaths"]))
log_step("Reconcile NOAA vs Wiki deaths", "USGS+NOAA+Wiki", int(disagree_mask.sum() + one_src.sum()), int(disagree_mask.sum()), 0, 
         "Reconciled cross-source disagreements via average, and kept single-source positive tolls")

# 4. Join World Bank Data
wb_cols = ["country_iso3", "year", "region", "income_group", "gdp_per_capita_ppp_constant_2021", 
           "population_total", "population_density_per_sq_km"]
df_master = usgs_noaa.merge(worldbank[wb_cols], on=["country_iso3", "year"], how="left")

# 5. Export Master Parquet and Cleaning Log
out_file = "../clean/earthquakes_master_2000_2025.parquet"
df_master.drop(columns=["date_dt"], inplace=True)
df_master.to_parquet(out_file, index=False)

master_log = pd.concat([pd.read_csv(f) for f in glob.glob("../cleaning_log_*.csv")] + [pd.DataFrame(cleaning_log)], ignore_index=True)
master_log["step"] = range(1, len(master_log) + 1)
master_log.to_csv("../cleaning_log.csv", index=False)
print(f"Master file saved to {out_file} | Shape: {df_master.shape}")

FileNotFoundError: [Errno 2] No such file or directory: '../interim/usgs_noaa_cleaned_2026-10-21.parquet'

In [ ]:
DICT_META = {
    "id": ("USGS event ID", "string", "USGS ComCat API", "Unique alphanumeric string"),
    "date": ("UTC date", "YYYY-MM-DD", "USGS / NOAA / Wiki", "2000-01-01 to 2025-12-31"),
    "year": ("Calendar year", "year", "USGS / World Bank", "2000 to 2025"),
    "latitude": ("Epicentre latitude", "decimal degrees", "USGS ComCat API", "-90.0 to 90.0"),
    "longitude": ("Epicentre longitude", "decimal degrees", "USGS ComCat API", "-180.0 to 180.0"),
    "depth": ("Focal depth", "km", "USGS ComCat API", "0.0 to 700.0 km"),
    "mag": ("Earthquake magnitude", "Mw / magnitude", "USGS ComCat API", "6.0 to 9.5"),
    "magType_clean": ("Magnitude scale", "categorical", "USGS ComCat API", "Mw, mb, ms, md, ml"),
    "country_clean": ("Standardized country name", "string", "Natural Earth / USGS", "Sovereign country or Offshore"),
    "country_iso3": ("ISO-3166 alpha-3 code", "ISO-3 code", "Natural Earth / WB", "3-letter uppercase or OFFSHORE"),
    "noaa_deaths": ("NOAA reported deaths", "persons (count)", "NOAA NCEI API", ">= 0"),
    "wiki_deaths": ("Wikipedia reported deaths", "persons (count)", "Wikipedia", ">= 0"),
    "deaths_disagree": ("NOAA vs Wiki disagreement flag", "boolean", "Derived", "True, False"),
    "deaths_final": ("Reconciled final death toll", "persons (count)", "NOAA + Wikipedia", ">= 0"),
    "is_default_depth": ("Fixed placeholder depth flag", "boolean", "Derived (USGS)", "True (10/33 km), False"),
    "cluster_shock_count": ("Shocks collapsed in cluster", "count", "Derived (USGS)", ">= 1"),
    "region": ("World Bank geographic region", "categorical", "World Bank WDI", "7 global regions"),
    "income_group": ("World Bank income tier", "categorical", "World Bank WDI", "Low, Lower-middle, Upper-middle, High"),
    "gdp_per_capita_ppp_constant_2021": ("Real PPP GDP per capita", "2021 intl $", "World Bank WDI", "> 0 intl $"),
    "population_total": ("Total national population", "persons", "World Bank WDI", "> 0"),
    "population_density_per_sq_km": ("Population density", "persons / sq km", "World Bank WDI", "> 0")
}

dict_rows = []
for col in df_master.columns:
    desc, unit, src, rng = DICT_META.get(col, (col, "numeric/text", "Derived", "Valid domain"))
    dict_rows.append({"column": col, "dtype": str(df_master[col].dtype), "unit": unit,
                      "source": src, "allowed_range": rng, "description": desc})

pd.DataFrame(dict_rows).to_csv("../data_dictionary.csv", index=False)
print("Saved data_dictionary.csv to project root.")

Saved data_dictionary.csv to project root.
